# HR Policy RAG Walkthrough

This beginner notebook demonstrates the Retrieval-Augmented Generation (RAG) pipeline in `app.py`. It reads a policy document, makes searchable chunks, retrieves relevant text, and asks an OpenAI chat model to answer using those passages.

> The included `Sample_HR_policy_Document.txt" ` is a synthetic example. To use your own source, follow the instructions in data/README.md `. Keep private policy documents and API keys out of GitHub.

## 1. Install packages and configure the API key

From the project folder, activate your virtual environment and install `requirements.txt`. Copy `.env.example` to `.env` and add your OpenAI API key there. The `.env` file is ignored by Git.

In [1]:
import os
from dotenv import load_dotenv
import json
import chromadb
from pathlib import Path
import pandas as pd
from openai import OpenAI


In [2]:
BASE_DIR = Path.cwd()
env_path = BASE_DIR / ".env"
print("Found .env:", env_path.exists())

load_dotenv(dotenv_path=env_path, override=True)

api_key = os.getenv("OPENAI_API_KEY")
print("Key loaded:", bool(api_key))  # Shows True or False, not the key

Found .env: True
Key loaded: True


In [3]:
client = OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")

## 2. Load the policy document

TXT_PATH = BASE_DIR / "data" / "Sample_HR_policy_Document.txt" selects the synthetic demo policy so the project works immediately. `load_document()` function reads the text file and returns its content as a string. If the file doesn't exist, it raises a `FileNotFoundError`.

In [ ]:
TXT_PATH = BASE_DIR / "data" / "Sample_HR_policy_Document.txt"
if not TXT_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {TXT_PATH}\n"
        "Place Sample_HR_policy_Document.txt inside the data folder."
    )

with open(TXT_PATH, "r", encoding="utf-8") as f:
    hr_document = f.read()

print(f"Characters: {len(hr_document)}")
print(f"Words: {len(hr_document.split())}")
print(hr_document[:500])

Characters: 19036
Words: 2589
Human Rights Policy		
Copyright © 2024, Tech Mahindra. All rights reserved.		
		
1.   INTRODUCTION		
This document specifies the “Human Rights Policy” of Tech Mahindra Ltd., its Subsidiaries and Joint Ventures  (hereinafter  referred  to  as  “Tech  Mahindra”  /  “TechM”  /  “Company”).  Respect  for  Human rights  is  a  fundamental  value  of  Tech  Mahindra.  We  strive  to  respect  and  promote  human  rights across all our global sites, in accordance with the UN Guiding Principles (UNGC) o


## 3. Split the document into chunks

A language model should receive only the most relevant parts of a long document. We split the policy into smaller passages; Chroma will later turn those passages into vectors for semantic search

In [6]:
def chunk_document(text, source_name):
    paragraphs = text.strip().split("\n") # We are splitting data with two new lines

    chunks = []
    for para in paragraphs:
        para = para.strip()

        if len(para) < 50:
            continue

        if para.startswith("====="):
            continue
        chunks.append({"text": para, "source": source_name})
    return chunks


all_chunks = chunk_document(hr_document, "HR Policy")

print(f"Total chunks: {len(all_chunks)}")
print("Sample chunk:")
print(f"  Source: {all_chunks[2]['source']}")
print(f"  Text: {all_chunks[2]['text'][:100]}...")

Total chunks: 75
Sample chunk:
  Source: HR Policy
  Text: We conduct due diligence to identify and prevent human rights risks to people in our business and va...


## 4. Store chunks in ChromaDB

`build_collection()` creates a local persistent Chroma database, removes stale chunks from a previous run, and indexes the current document. The generated `chroma_db/` folder is excluded from Git because it can be recreated.

In [7]:
chroma_client = chromadb.Client()

In [8]:
collection = chroma_client.get_or_create_collection(name="company_docs")

documents = []
ids = []
metadatas = []

for i, chunk in enumerate(all_chunks):
    documents.append(chunk["text"])
    ids.append(f"chunk_{i}")
    metadatas.append({"source": chunk["source"]})

collection.add(documents = documents,
               ids = ids,
               metadatas = metadatas)

print(f"Stored {len(documents)} chunks in Chroma DB")

Stored 75 chunks in Chroma DB


## 5. Retrieve relevant policy passages

Retrieval runs before generation. It finds the passages most similar in meaning to the question and returns their source labels.

In [9]:
def retrieve(question, n_results = 3):
    results = collection.query(query_texts = [question],
                 n_results = n_results)
    return results["documents"][0], results["metadatas"][0]

In [10]:
chunks, sources = retrieve("What is the work from home policy?", 3)
for i in range(len(chunks)):
    print(f"── Chunk {i+1} ──")
    print(f"Source: {sources[i]['source']}")
    print(f"Text: {chunks[i]}")
    print()


── Chunk 1 ──
Source: HR Policy
Text: Flexible work arrangements are provided across all TechM locations including benefits like Parental Leave, Paternity and Maternity leave, Sabbatical for Higher Education, Time off for personal reasons and Work from Home (WFH) as per business requirement and under the prevailing regulation. WFH ensures continuity of our services during adverse scenarios and helps safeguard the health, safety, and welfare of our associates, which is of paramount importance to us. We also provide time off for Individual Corporate Responsibility and Corporate Social Responsibility activities."

── Chunk 2 ──
Source: HR Policy
Text: The policy is reviewed at least once a year or as and when there are any changes required.

── Chunk 3 ──
Source: HR Policy
Text: instructions,  and  training  programs  to  enable  all  associates  to  comply  with  the  Health,  Safety  and Environment policy of the company."



In [11]:
context = "\n\n".join(chunks)
print(context)

Flexible work arrangements are provided across all TechM locations including benefits like Parental Leave, Paternity and Maternity leave, Sabbatical for Higher Education, Time off for personal reasons and Work from Home (WFH) as per business requirement and under the prevailing regulation. WFH ensures continuity of our services during adverse scenarios and helps safeguard the health, safety, and welfare of our associates, which is of paramount importance to us. We also provide time off for Individual Corporate Responsibility and Corporate Social Responsibility activities."

The policy is reviewed at least once a year or as and when there are any changes required.

instructions,  and  training  programs  to  enable  all  associates  to  comply  with  the  Health,  Safety  and Environment policy of the company."


## 6. Generate a grounded answer

This final step sends the retrieved passages and question to the configured OpenAI model. It may use API credits. The system instruction asks the model to say when the retrieved policy text does not contain an answer.

In [12]:
def ask_rag(question, n_results = 3, verbose = True):

    chunks, scores = retrieve(question, n_results)

    if verbose:
        print(f"\n{'═' * 60}")
        print(f" Question: {question}")
        print(f"{'─' * 60}")
        print(f" Retrieved {len(chunks)} chunks:")
        for i, (chunk, source) in enumerate(zip(chunks, sources)):
            print(f"   [{source['source']}] {chunk[:80]}...")
        print(f"{'─' * 60}")


    context = "\n\n".join(chunks)

    messages = [
        {
            "role": "system",
            "content": ("you are an helpful assistant that answers questions based only on the provided context."
                        "If the context does not contain enough informaiton to answer the question"
                        "say I dont have enough contet or informaiton to asnwer this quesiton"
                        "Do Not I repat Do Not make up information. Always be conscise."
                        "if it is a general quesiton like captials of country, you can answer from your knowledge"
            )
    },
        {
            "role": "user",
            "content" : f"Context:\n{context}\n\n Question: {question}"
        }
        ]

    response = client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages = messages,
                temperature = 0.2,
                # tools = []
            )


    answer = response.choices[0].message.content

    if verbose:
        print(f" Answer: {answer}")
        print(f"{'═' * 60}")

    return answer

print("✅ RAG pipeline ready!")

✅ RAG pipeline ready!


In [13]:
ask_rag("What is the work from home policy?")


════════════════════════════════════════════════════════════
 Question: What is the work from home policy?
────────────────────────────────────────────────────────────
 Retrieved 3 chunks:
   [HR Policy] Flexible work arrangements are provided across all TechM locations including ben...
   [HR Policy] The policy is reviewed at least once a year or as and when there are any changes...
   [HR Policy] instructions,  and  training  programs  to  enable  all  associates  to  comply ...
────────────────────────────────────────────────────────────
 Answer: The Work‑From‑Home (WFH) policy at TechM allows associates at all locations to work from home **when required by business needs and in accordance with applicable regulations**. It is intended to:

* Ensure continuity of services during adverse or disruptive situations.  
* Protect the health, safety and welfare of employees.  

The policy is reviewed at least once a year (or whenever changes are needed).
═══════════════════════════════════

'The Work‑From‑Home (WFH) policy at TechM allows associates at all locations to work from home **when required by business needs and in accordance with applicable regulations**.\u202fIt is intended to:\n\n* Ensure continuity of services during adverse or disruptive situations.  \n* Protect the health, safety and welfare of employees.  \n\nThe policy is reviewed at least once a year (or whenever changes are needed).'

## What to try next

- Ask about leave, confidentiality, or respectful workplace expectations in the demo policy.
- Put your own authorized policy in `data/`.
- Change `n_results` to compare how the number of retrieved passages affects the response.
